# Simulating Human Sleep-Stage Transitions and Nighttime Sleep Fragmentatio

This notebook develops a four-state, 30-second sleep-stage model from manually scored Sleep-EDF Expanded hypnograms. This milestone downloads a fixed pilot subset, prepares reproducible training and held-out data, and checks the transition counts.

## 1. Problem and question

How does increasing the probability of nighttime awakenings affect expected Deep Sleep and REM duration? The planned study will compare an empirical baseline with hypothetical changes in sleep-to-Wake transition probabilities, measuring stage minutes and fragmentation across replicated nights.

## 2. Model, assumptions and equations

Let $X_t$ be the stage in epoch $t$, where each epoch is 30 seconds and the possible states are Wake, Light, Deep and REM. The first-order, time-homogeneous DTMC assumes

$$P_{ij}=\Pr(X_{t+1}=j\mid X_t=i),\qquad \sum_j P_{ij}=1,\qquad \pi_{t+1}=\pi_t P.$$

Here $\pi_t$ is the row vector of state probabilities. Fit each dimensionless transition probability from training counts $C_{ij}$ as $\widehat P_{ij}=C_{ij}/\sum_k C_{ik}$ for rows with observed departures (including self-transitions). Estimate the initial distribution from the observed stages at the starts of training windows. A future simulator will sample from each row's cumulative probabilities using uniform random variates.

For a sleep-state row, if the baseline Wake probability is $q_i=P_{iW}<1$ and the scenario uses $q'_i$, set $P'_{iW}=q'_i$ and $P'_{ij}=P_{ij}(1-q'_i)/(1-q_i)$ for $j\ne W$. The Wake row stays fixed. A stage's total duration in minutes is its epoch count multiplied by $0.5$. An awakening is a sleep-to-Wake transition after sleep onset. The planned experiment uses independent night replications for means and 95% Monte Carlo confidence intervals.

The model pools training participants and assumes the same transition law throughout the analysis window. A homogeneous DTMC implies geometric stage run lengths. The grouped labels are derived from the historical Rechtschaffen–Kales scores. The scenarios are hypothetical model perturbations.

**Concepts applied** — DTMC state evolution, empirical input modelling and discrete inverse-transform sampling, and Monte Carlo estimation with the law of large numbers.

## 3. Implementation

In Colab, choose **Runtime → Run all**. The first code cell installs the EDF+ reader. The notebook downloads only the annotation files from the versioned PhysioNet source.

The fixed pilot manifest includes both listed nights for the first 12 Sleep Cassette participant IDs. A seeded participant split reserves three participants for later validation. This convenience subset keeps development repeatable and manageable. Its representativeness is unknown.

The provisional night window extends up to 30 minutes before the first and after the last scored non-Wake epoch, clipped to the annotated span. The padding follows the approach described in the MNE-Python sleep tutorial [3]. The same rule applies to every record. Original epoch indices remain relative to the recording. Movement, unscored intervals and uncovered epochs remain explicitly invalid. Only adjacent valid epochs within one recording contribute to transition counts.

In [1]:
%pip install --quiet mne==1.10.2

In [2]:
from hashlib import sha256
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.error import URLError
from urllib.request import urlopen

import mne
import numpy as np
import pandas as pd
from IPython.display import display

SOURCE = "https://physionet.org/files/sleep-edfx/1.0.0/sleep-cassette/"
MIRROR = "https://physionet-open.s3.amazonaws.com/sleep-edfx/1.0.0/sleep-cassette/"
RECORDS = """
SC4001EC SC4002EC SC4011EH SC4012EC SC4021EH SC4022EJ
SC4031EC SC4032EP SC4041EC SC4042EC SC4051EC SC4052EC
SC4061EC SC4062EC SC4071EC SC4072EH SC4081EC SC4082EP
SC4091EC SC4092EC SC4101EC SC4102EC SC4111EC SC4112EC
""".split()
STATES = ("Wake", "Light", "Deep", "REM")
STAGE_MAP = {
    "Sleep stage W": "Wake",
    "Sleep stage 1": "Light",
    "Sleep stage 2": "Light",
    "Sleep stage 3": "Deep",
    "Sleep stage 4": "Deep",
    "Sleep stage R": "REM",
    "Movement time": None,
    "Sleep stage ?": None,
}
EPOCH_SECONDS = 30
PADDING_EPOCHS = 30 * 60 // EPOCH_SECONDS
SPLIT_SEED = 4042

participants = sorted({record[:5] for record in RECORDS})
held_out = set(np.random.default_rng(SPLIT_SEED).choice(participants, size=3, replace=False))

In [3]:
def expand_night(annotations):
    """Expand scored spans onto the recording's epoch grid and crop the window."""
    raw_labels = {}
    for onset, duration, label in zip(
        annotations.onset, annotations.duration, annotations.description, strict=True
    ):
        if label not in STAGE_MAP:
            raise ValueError(f"Unexpected Sleep-EDF annotation: {label!r}")
        start = round(onset / EPOCH_SECONDS)
        length = round(duration / EPOCH_SECONDS)
        if (
            start < 0
            or length <= 0
            or not np.isclose(onset, start * EPOCH_SECONDS, rtol=0, atol=1e-6)
            or not np.isclose(duration, length * EPOCH_SECONDS, rtol=0, atol=1e-6)
        ):
            raise ValueError(f"Invalid annotation timing: {onset=}, {duration=}, {label=}")
        for epoch in range(start, start + length):
            if epoch in raw_labels:
                raise ValueError(f"Overlapping annotations at epoch {epoch}")
            raw_labels[epoch] = label

    sleep = [
        epoch for epoch, label in raw_labels.items()
        if STAGE_MAP[label] in ("Light", "Deep", "REM")
    ]
    if not sleep:
        raise ValueError("No scored sleep in recording")
    first = max(min(raw_labels), min(sleep) - PADDING_EPOCHS)
    last = min(max(raw_labels), max(sleep) + PADDING_EPOCHS)
    night = pd.DataFrame({"epoch": range(first, last + 1)})
    night["label"] = night["epoch"].map(raw_labels).fillna("Uncovered")
    night["stage"] = night["label"].map(STAGE_MAP)
    return night


def count_transitions(night):
    """Count adjacent pairs with two valid stages in one recording."""
    following = night["stage"].shift(-1)
    adjacent = night["epoch"].shift(-1).sub(night["epoch"]).eq(1)
    counts = pd.crosstab(night.loc[adjacent, "stage"], following.loc[adjacent])
    return counts.reindex(index=STATES, columns=STATES, fill_value=0).astype("int64")


def fetch_annotation(filename):
    errors = []
    for base_url in (SOURCE, MIRROR):
        url = base_url + filename
        try:
            with urlopen(url, timeout=15) as response:
                return response.read(), response.geturl()
        except (URLError, TimeoutError) as exc:
            errors.append(f"{url}: {exc}")
    raise RuntimeError("Could not download annotation. " + " | ".join(errors))


nights, counts_by_record, audit_rows = {}, {}, []
with TemporaryDirectory(prefix="sleep-edf-") as temp:
    for record in RECORDS:
        participant = record[:5]
        split = "held-out" if participant in held_out else "training"
        filename = f"{record}-Hypnogram.edf"
        contents, url = fetch_annotation(filename)
        try:
            path = Path(temp) / filename
            path.write_bytes(contents)
            night = expand_night(mne.read_annotations(path))
        except (OSError, ValueError, RuntimeError) as exc:
            raise RuntimeError(f"Could not process {url}: {exc}") from exc

        counts = count_transitions(night)
        nights[record] = night
        counts_by_record[record] = counts
        audit_rows.append({
            "record": filename,
            "participant": participant,
            "split": split,
            "source_url": url,
            "sha256": sha256(contents).hexdigest(),
            "first_epoch": night["epoch"].iloc[0],
            "last_epoch": night["epoch"].iloc[-1],
            "window_epochs": len(night),
            "valid_epochs": night["stage"].notna().sum(),
            **{state: night["stage"].eq(state).sum() for state in STATES},
            "movement": night["label"].eq("Movement time").sum(),
            "unscored": night["label"].eq("Sleep stage ?").sum(),
            "uncovered": night["label"].eq("Uncovered").sum(),
            "valid_pairs": counts.to_numpy().sum(),
        })

audit = pd.DataFrame(audit_rows)
split_counts = {
    split: sum(counts_by_record[record] for record in RECORDS
               if (record[:5] in held_out) == (split == "held-out"))
    for split in ("training", "held-out")
}

## 4. Verification

Check that the split is disjoint, all window epochs are accounted for, and the count of valid pairs equals the number of valid epochs minus the number of contiguous valid runs, calculated separately for each recording.

In [4]:
assert STAGE_MAP["Sleep stage W"] == "Wake"
assert STAGE_MAP["Sleep stage 1"] == "Light"
assert STAGE_MAP["Sleep stage 2"] == "Light"
assert STAGE_MAP["Sleep stage 3"] == "Deep"
assert STAGE_MAP["Sleep stage 4"] == "Deep"
assert STAGE_MAP["Sleep stage R"] == "REM"
assert STAGE_MAP["Movement time"] is None
assert STAGE_MAP["Sleep stage ?"] is None
print("Sleep stage mapping checks passed.")

# Valid pairs on either side of a missing epoch should still count.
night_with_gap = pd.DataFrame({
    "epoch": range(5),
    "stage": ["Wake", "Light", None, "Deep", "REM"],
})
gap_counts = count_transitions(night_with_gap)
assert gap_counts.loc["Wake", "Light"] == 1
assert gap_counts.loc["Deep", "REM"] == 1
assert gap_counts.loc["Light", "Deep"] == 0
assert gap_counts.to_numpy().sum() == 2
print("Missing-stage transition check passed.")

night_with_skipped_epoch = pd.DataFrame({
    "epoch": [0, 1, 3, 4],
    "stage": ["Wake", "Light", "Deep", "REM"],
})
skipped_epoch_counts = count_transitions(night_with_skipped_epoch)
assert skipped_epoch_counts.loc["Wake", "Light"] == 1
assert skipped_epoch_counts.loc["Deep", "REM"] == 1
assert skipped_epoch_counts.loc["Light", "Deep"] == 0
assert skipped_epoch_counts.to_numpy().sum() == 2
print("Non-consecutive epoch transition check passed.")

assert len(audit) == len(RECORDS) == len(set(RECORDS))
assert set(audit["split"]) == {"training", "held-out"}
training_participants = set(audit.loc[audit["split"].eq("training"), "participant"])
held_out_participants = set(audit.loc[audit["split"].eq("held-out"), "participant"])
assert training_participants.isdisjoint(held_out_participants), "Training and held-out participants overlap"
assert audit.groupby("participant")["split"].nunique().eq(1).all()
assert audit.loc[audit["split"].eq("held-out"), "participant"].nunique() == len(held_out)
repeated_held_out = set(np.random.default_rng(SPLIT_SEED).choice(
    participants, size=len(held_out), replace=False
))
assert repeated_held_out == held_out_participants, "Held-out split did not repeat with the same seed"
assert set(participants) - repeated_held_out == training_participants, "Training split did not repeat with the same seed"
print("Participant split reproducibility check passed.")
assert audit[list(STATES)].sum(axis=1).eq(audit["valid_epochs"]).all()
assert audit[["valid_epochs", "movement", "unscored", "uncovered"]].sum(axis=1).eq(
    audit["window_epochs"]
).all()
for record, night in nights.items():
    assert night["epoch"].diff().iloc[1:].eq(1).all()
    valid = night["stage"].notna()
    run_starts = valid & ~valid.shift(fill_value=False)
    assert counts_by_record[record].to_numpy().sum() == valid.sum() - run_starts.sum()
for split, counts in split_counts.items():
    assert counts.to_numpy().sum() == audit.loc[audit["split"].eq(split), "valid_pairs"].sum()
training_row_totals = split_counts["training"].sum(axis=1)
empty_stages = training_row_totals.index[training_row_totals.le(0)].tolist()
assert not empty_stages, f"No outgoing training transitions for: {', '.join(empty_stages)}"
print("Preprocessing checks passed.")

Preprocessing checks passed.


## 5. Experiments and results

The tables audit the preprocessing inputs and transition counts. Rows in the count matrices are source stages and columns are destination stages. Held-out counts are kept separate from training. The planned scenario replications and confidence intervals follow in a later milestone.

In [5]:
print(f"Source: {SOURCE}")
print(f"Participant split seed: {SPLIT_SEED}")
display(audit)
display(audit.groupby("split").agg(
    participants=("participant", "nunique"),
    recordings=("record", "size"),
    window_epochs=("window_epochs", "sum"),
    valid_epochs=("valid_epochs", "sum"),
    movement=("movement", "sum"),
    unscored=("unscored", "sum"),
    uncovered=("uncovered", "sum"),
    valid_pairs=("valid_pairs", "sum"),
))
for split, counts in split_counts.items():
    print(f"{split}: observed transition counts")
    display(counts)

Source: https://physionet.org/files/sleep-edfx/1.0.0/sleep-cassette/
Participant split seed: 4042


,record,participant,split,source_url,sha256,first_epoch,last_epoch,window_epochs,valid_epochs,Wake,Light,Deep,REM,movement,unscored,uncovered,valid_pairs
0,SC4001EC-Hypnogram.edf,SC400,training,https://physionet-open.s3.amazonaws.com/sleep-...,a4cf67694ade1b52a0ddd06d5817fd45d2d3e8bac5302f...,961,1801,841,841,188,308,220,125,0,0,0,840
1,SC4002EC-Hypnogram.edf,SC400,training,https://physionet-open.s3.amazonaws.com/sleep-...,76e8457a67eaaaa62c7f6cc6e098fbe1c0a9c3fa02fa90...,809,1936,1128,1127,183,432,297,215,1,0,0,1125
2,SC4011EH-Hypnogram.edf,SC401,held-out,https://physionet-open.s3.amazonaws.com/sleep-...,5bbf7b4ed8a95c8d20e4347b08c2ded2ee21e2dac4680d...,658,1760,1103,1103,157,671,105,170,0,0,0,1102
3,SC4012EC-Hypnogram.edf,SC401,held-out,https://physionet-open.s3.amazonaws.com/sleep-...,b6323df8b3867a5fdbd80f0cd8233b59659ac496fa48c8...,583,1768,1186,1186,162,752,96,176,0,0,0,1185
4,SC4021EH-Hypnogram.edf,SC402,training,https://physionet-open.s3.amazonaws.com/sleep-...,62b49b26bf08941cb0e974cfeeb0a820bf7dd43fb0c66d...,669,1693,1025,1025,128,639,95,163,0,0,0,1024
5,SC4022EJ-Hypnogram.edf,SC402,training,https://physionet-open.s3.amazonaws.com/sleep-...,5289ca8a9417307cbbf8b2a4e511909ac742c69d45c17a...,697,1706,1010,1009,125,586,119,179,1,0,0,1007
6,SC4031EC-Hypnogram.edf,SC403,training,https://physionet-open.s3.amazonaws.com/sleep-...,7ec3d3c8d8f1a89d48df4cc85be374fbdf70d4b5ec5ec3...,811,1762,952,952,140,546,57,209,0,0,0,951
7,SC4032EP-Hypnogram.edf,SC403,training,https://physionet-open.s3.amazonaws.com/sleep-...,0ef073c5cfba31459732be96cc51e7106d5650c1db1b96...,861,1771,911,911,136,445,131,199,0,0,0,910
8,SC4041EC-Hypnogram.edf,SC404,held-out,https://physionet-open.s3.amazonaws.com/sleep-...,da33d90216e031a6b04c9ebaddcdf7abc223f1e61fd589...,752,1987,1236,1235,200,786,53,196,1,0,0,1233
9,SC4042EC-Hypnogram.edf,SC404,held-out,https://physionet-open.s3.amazonaws.com/sleep-...,28ece9383d5c49f375a34d63c1226cd0eb8c5a4bb9f887...,1014,2217,1204,1200,185,651,94,270,4,0,0,1195


,participants,recordings,window_epochs,valid_epochs,movement,unscored,uncovered,valid_pairs
split,,,,,,,,
held-out,3,6,6917,6912,5,0,0,6901
training,9,18,18083,18053,30,0,0,18006


training: observed transition counts


stage,Wake,Light,Deep,REM
stage,,,,
Wake,2824,189,1,4
Light,147,8762,355,170
Deep,9,342,2007,1
REM,36,136,0,3023


held-out: observed transition counts


stage,Wake,Light,Deep,REM
stage,,,,
Wake,1072,82,0,11
Light,65,3269,145,79
Deep,7,137,834,1
REM,21,69,1,1108


### Training transition counts

These counts come from the saved preprocessing results. Rows show the current stage and columns show the stage in the next 30-second epoch.

| Current stage | Wake | Light | Deep | REM |
| --- | ---: | ---: | ---: | ---: |
| Wake | 2824 | 189 | 1 | 4 |
| Light | 147 | 8762 | 355 | 170 |
| Deep | 9 | 342 | 2007 | 1 |
| REM | 36 | 136 | 0 | 3023 |

The training set contains 18,006 valid pairs of consecutive 30-second epochs from 18 recordings across nine participants. These counts will be used to estimate the model's transition probabilities. The held-out set contains 6,901 valid pairs from six recordings across three different participants and is reserved for validation.

In both tables, the most common pair is Light → Light, meaning two consecutive epochs were both scored as Light sleep.

The audit shows 30 movement epochs in the training set and 5 in the held-out set. Both sets had zero unscored and zero uncovered epochs. These categories have no usable sleep-stage label, so they break the sequence. A valid transition requires two consecutive epochs with valid stage labels.

### Training transition probabilities

Each probability is the count divided by its row total. Rows show the current stage and columns show the next stage. The table rounds probabilities to six decimal places.

In [ ]:
# Divide each count by its current-stage row total.
training_counts = split_counts["training"].loc[list(STATES), list(STATES)]
training_row_totals = training_counts.sum(axis=1)
empty_stages = training_row_totals.index[training_row_totals.le(0)].tolist()
if empty_stages:
    raise ValueError(f"No outgoing training transitions for: {', '.join(empty_stages)}")
training_transition_probabilities = training_counts.div(training_row_totals, axis="index")
probabilities = training_transition_probabilities.to_numpy()
assert ((probabilities >= 0) & (probabilities <= 1)).all(), "Transition probabilities must be between 0 and 1"
assert np.allclose(probabilities.sum(axis=1), 1), "Each transition probability row must sum to 1"
display(training_transition_probabilities.round(6))

,Wake,Light,Deep,REM
stage,,,,
Wake,0.935719,0.062624,0.000331,0.001325
Light,0.015582,0.928768,0.037630,0.018020
Deep,0.003815,0.144977,0.850784,0.000424
REM,0.011268,0.042567,0.000000,0.946166


### Initial sleep-stage probabilities

Each training recording contributes the first 30-second epoch of its preprocessed window. Windows with an unusable first-epoch stage are excluded from the denominator. The window can begin during the Wake padding before sleep onset.

In [ ]:
# Use the first epoch of each preprocessed training window.
training_first_stages = pd.Series(
    {f"{record}-Hypnogram.edf": night["stage"].iloc[0]
     for record, night in nights.items() if record[:5] not in held_out},
    dtype="object",
)
expected_training_records = audit.loc[audit["split"].eq("training"), "record"]
assert (
    len(training_first_stages) == len(expected_training_records)
    and set(training_first_stages.index) == set(expected_training_records)
), "Expected exactly one first epoch from each training recording and none from held-out recordings"
initial_stage_counts = training_first_stages.value_counts().reindex(list(STATES), fill_value=0)
valid_starts = int(initial_stage_counts.sum())
excluded_starts = len(training_first_stages) - valid_starts
if valid_starts == 0:
    raise ValueError("No training windows have a valid first-epoch sleep stage")

initial_stage_probabilities = initial_stage_counts / valid_starts
assert np.isclose(initial_stage_probabilities.sum(), 1), "Initial stage probabilities must sum to 1"
display(pd.DataFrame({
    "Training starts": initial_stage_counts,
    "Initial probability": initial_stage_probabilities.round(6),
}).rename_axis("Sleep stage"))
print(f"Valid training starts: {valid_starts} of {len(training_first_stages)}. Excluded starts: {excluded_starts}.")
print(f"Initial probability sum: {initial_stage_probabilities.sum():.6f}")

,Training starts,Initial probability
Sleep stage,,
Wake,18,1.0
Light,0,0.0
Deep,0,0.0
REM,0,0.0


Valid training starts: 18 of 18. Excluded starts: 0.
Initial probability sum: 1.000000


### Sample the next sleep stage

Draw the next 30-second stage from the current stage's row in the unrounded training matrix. Pass the same NumPy random generator at each step to continue its random sequence.

In [ ]:
def sample_next_stage(current_stage, transition_probabilities, rng):
    """Draw one next stage using the current stage's transition probabilities."""
    probabilities = transition_probabilities.loc[current_stage, list(STATES)].to_numpy()
    return rng.choice(STATES, p=probabilities)

### Check sampled transition frequencies

Draw 20,000 times from the Light row with a separate seeded generator. Each observed proportion should be within one percentage point of the corresponding training probability.

In [ ]:
frequency_rng = np.random.default_rng(4042)
sample_count = 20_000
draws = [
    sample_next_stage("Light", training_transition_probabilities, frequency_rng)
    for _ in range(sample_count)
]
observed_frequencies = pd.Series(draws).value_counts().reindex(list(STATES), fill_value=0) / sample_count
expected_frequencies = training_transition_probabilities.loc["Light", list(STATES)]
max_frequency_error = (observed_frequencies - expected_frequencies).abs().max()
assert max_frequency_error <= 0.01, f"Sampled frequencies differ by up to {max_frequency_error:.2%}"
print(f"Sampling frequency check passed. Largest difference: {max_frequency_error * 100:.2f} percentage points.")

### Simulate one eight-hour night

Sample the first stage from the training window-start distribution. Use each sampled stage to draw the next 30-second epoch with the same random generator.

In [ ]:
def simulate_night(initial_probabilities, transition_probabilities, rng):
    """Return the ordered stages for one eight-hour sleep window."""
    epoch_count = 8 * 60 * 60 // EPOCH_SECONDS
    starting_weights = initial_probabilities.loc[list(STATES)].to_numpy()
    stages = [rng.choice(STATES, p=starting_weights)]
    for _ in range(epoch_count - 1):
        stages.append(sample_next_stage(stages[-1], transition_probabilities, rng))
    return stages

## 6. Discussion and limitations

The first/last scored sleep rule is an approximation of the night window. Wake counts and estimated starting stages depend on the selected padding. Missing intervals interrupt transitions, so the matrices describe observed valid neighbours. The pilot is small, and pooled counts give more weight to recordings with more observed transitions. Later validation will compare simulated and held-out stage durations and awakening counts under consistent window and metric definitions.

## 7. References

1. B. Kemp, *Sleep-EDF Database Expanded*, v1.0.0, PhysioNet. [Dataset and annotation documentation](https://physionet.org/content/sleep-edfx/1.0.0/). DOI: 10.13026/C2X676. [ODC Attribution v1.0 licence](https://physionet.org/content/sleep-edfx/view-license/1.0.0/).
2. B. Kemp, A. H. Zwinderman, B. Tuk, H. A. C. Kamphuisen, and J. J. L. Oberyé, “Analysis of a sleep-dependent neuronal feedback loop: the slow-wave microcontinuity of the EEG,” *IEEE Transactions on Biomedical Engineering*, vol. 47, no. 9, pp. 1185–1194, 2000. [Publisher record](https://ieeexplore.ieee.org/document/867928/).
3. MNE-Python contributors, “Sleep stage classification from polysomnography (PSG) data.” [MNE-Python tutorial](https://mne.tools/stable/auto_tutorials/clinical/60_sleep.html#extract-30s-events-from-annotations).